# Module 6: Feature Store Concepts

**Snowpark ML Fundamentals - Week 2 | Lunch & Learn**

---

## Learning Objectives
- Understand what a Feature Store is and why it matters
- Learn the Snowflake Feature Store architecture
- Generate synthetic temporal data for feature engineering
- Explore the data patterns that drive feature creation

## Key Concept
> A **Feature Store** is a centralized repository for storing, sharing, and
> serving ML features. It ensures **consistency** between training and inference,
> enables **reuse** across models, and provides **governance** over feature definitions.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 6.1 What is a Feature Store?

| Problem | Feature Store Solution |
|---------|------------------------|
| Training/serving skew | Single source of truth for feature definitions |
| Feature duplication | Centralized catalog with discovery |
| Point-in-time correctness | Temporal joins prevent data leakage |
| Governance | Entity-based access control, lineage tracking |

### Snowflake Feature Store Architecture

```
                        SNOWFLAKE WAREHOUSE
  ====================================================================

   RAW DATA               FEATURE ENGINEERING        FEATURE STORE
  +----------------+     +--------------------+     +----------------+
  | Transactions   |     | SQL / dbt / Snowpark|     | Entities       |
  | Interactions   | --> | - Time windows      | --> | Feature Views  |
  | Events         |     | - Aggregations      |     | Training Sets  |
  +----------------+     | - Ratios & buckets  |     +-------+--------+
                          | - Deduplication     |             |
                          +--------------------+             |
                                                    +--------+--------+
                                                    |                 |
                                                    v                 v
                                              +-----------+   +-----------+
                                              | TRAINING  |   | INFERENCE |
                                              |           |   |           |
                                              | Consistent|   | Latest    |
                                              | features  |   | features  |
                                              | + labels  |   | + model   |
                                              +-----------+   +-----------+

  ====================================================================
```

## 6.2 Setup & Data Generation

In [2]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session

session = create_session(env_path='../.env')
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

Connected: "MLDS_Q"."PREDICTIONS"
Warehouse: "TASK_WH"


In [3]:
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
)

# Generate synthetic temporal data
transactions_df = create_customer_transactions_dataset(session, n_rows=50000)
interactions_df = create_customer_interactions_dataset(session, n_rows=100000)

print(f"Transactions: {transactions_df.count()} rows")
print(f"Interactions: {interactions_df.count()} rows")

Transactions: 50000 rows
Interactions: 100000 rows


You can validate at this point that we have data
```sql
use role MLDS_ROLE_Q;
use database MLDS_Q;
use schema PREDICTIONS;

select * from CUSTOMER_TRANSACTIONS;
select * from CUSTOMER_INTERACTIONS;
```

## 6.3 Explore the Temporal Data

In [4]:
# Transaction data — note the ORDER_DATE column for time-windowed features
transactions_df.show(5)

------------------------------------------------------------------------------------------------------------------------------
|"TRANSACTION_ID"  |"CUSTOMER_ID"  |"ORDER_DATE"  |"ORDER_AMOUNT"  |"CATEGORY"   |"ORDER_STATUS"  |"ITEM_COUNT"  |"CHANNEL"  |
------------------------------------------------------------------------------------------------------------------------------
|0                 |1911           |2025-05-20    |3174.0          |TRAVEL       |COMPLETED       |9             |IN_STORE   |
|1                 |123            |2025-07-01    |4370.0          |HOME         |COMPLETED       |8             |WEB        |
|2                 |148            |2025-11-23    |3327.0          |SPORTS       |COMPLETED       |10            |IN_STORE   |
|3                 |1942           |2025-10-07    |1615.0          |SPORTS       |COMPLETED       |9             |MOBILE     |
|4                 |387            |2024-04-10    |1034.0          |ELECTRONICS  |COMPLETED       |7           

In [5]:
# Interaction data — note the INTERACTION_DATE and INTERACTION_TYPE
interactions_df.show(5)

---------------------------------------------------------------------------------------------------------------
|"INTERACTION_ID"  |"CUSTOMER_ID"  |"INTERACTION_DATE"  |"INTERACTION_TYPE"  |"CHANNEL"  |"DURATION_SECONDS"  |
---------------------------------------------------------------------------------------------------------------
|0                 |1911           |2025-10-10          |EMAIL_OPEN          |EMAIL      |124                 |
|1                 |123            |2025-10-31          |EMAIL_CLICK         |MOBILE     |120                 |
|2                 |148            |2026-01-12          |EMAIL_OPEN          |MOBILE     |132                 |
|3                 |1942           |2025-12-19          |CLICK               |MOBILE     |40                  |
|4                 |387            |2025-03-22          |CLICK               |WEB        |132                 |
--------------------------------------------------------------------------------------------------------

In [6]:
# Date range of transactions
from snowflake.snowpark import functions as F

transactions_df.select(
    F.min("ORDER_DATE").alias("EARLIEST"),
    F.max("ORDER_DATE").alias("LATEST"),
    F.count_distinct("CUSTOMER_ID").alias("UNIQUE_CUSTOMERS"),
).show()

------------------------------------------------
|"EARLIEST"  |"LATEST"    |"UNIQUE_CUSTOMERS"  |
------------------------------------------------
|2024-03-03  |2026-03-02  |2000                |
------------------------------------------------



In [7]:
# Interaction type distribution
interactions_df.group_by("INTERACTION_TYPE").agg(
    F.count("*").alias("COUNT")
).sort("COUNT", ascending=False).show()

--------------------------------
|"INTERACTION_TYPE"  |"COUNT"  |
--------------------------------
|SUPPORT_TICKET      |20099    |
|EMAIL_CLICK         |19999    |
|PAGE_VIEW           |19981    |
|CLICK               |19977    |
|EMAIL_OPEN          |19944    |
--------------------------------



## Key Takeaways

1. A **Feature Store** centralizes feature definitions for consistency and reuse
2. Snowflake's Feature Store uses **Entities** (join keys) and **Feature Views** (feature definitions)
3. **Temporal data** is the foundation — dates enable time-windowed aggregations
4. All data generation runs **server-side** in Snowflake (no local computation)

---

**Next: [07 - Feature Engineering SQL](07_feature_engineering_sql.ipynb)**

In [8]:
session.close()